# Identifying Strategic Investment Opportunities in Australian Olympic Sports
### A Data Analytics Case Study: Youth Participation × Media Visibility Ahead of Brisbane 2032

**Author:** Pratham Shah  
**Data sources:** AusPlay By-Sport Data Tables (Nov 2025) · Guardian Australia API  
**Tools:** Python · pandas · scikit-learn (TF-IDF, K-Means) · Plotly


## Executive Summary

The Australian Sports Commission (ASC) faces a resource-allocation problem ahead of Brisbane 2032: which Olympic sports should receive investment to maximise long-term fan engagement among Australian youth?

This project answers that question by joining two evidence streams:

- **Structured data** — AusPlay participation rates across four sheets (child totals, age bands, year-on-year change, organised sport enrolment) for all Olympic-aligned sports
- **Semi-structured data** — Guardian Australia sport-section articles collected via the Guardian Open Platform API, vectorised with TF-IDF and used to measure media visibility per sport

The analysis progresses through four stages: exploratory participation ranking → TF-IDF media landscape → composite opportunity scoring → K-Means strategic segmentation. The key finding is a consistent participation-visibility gap: **Gymnastics, Hockey, Volleyball, Surfing, and Taekwondo** each have tens of thousands of young Australian participants but appear in fewer than seven Guardian articles across a 170-article corpus. These are the highest-leverage targets for ASC investment.


## Research Question

> *Which Olympic sports show strong youth participation but low media visibility, and where could the Australian Sports Commission most effectively invest to build long-term fan engagement ahead of Brisbane 2032?*

### Why this question matters

Initial exploratory analysis (see the companion notebook `01_participation_exploration.ipynb`) showed that simply ranking sports by youth participation reproduces a list already dominated by Swimming, Football, and Basketball — sports that already attract the bulk of media attention, sponsorship, and athlete funding. Investing further in those sports is unlikely to produce marginal gains for fan development.

The more useful question is: **where does a participation base already exist but has not yet been connected to the Olympic programme?** Answering that requires measuring both participation *and* media visibility — neither alone is sufficient.


## Analytical Approach

Two advanced text and machine-learning techniques are combined:

| Technique | Purpose |
|-----------|----------|
| **TF-IDF vectorisation** | Converts Guardian article text into weighted term-frequency vectors; measures which sports and themes dominate the media corpus |
| **K-Means clustering** | Groups Olympic sports by their combined participation-and-visibility profile into four actionable strategic segments |

TF-IDF was chosen over raw term counts because it down-weights words that appear in nearly every article (`sport`, `australian`) and surfaces terms distinctive to specific sub-topics. K-Means was chosen because the goal is exploratory segmentation rather than prediction.

### Use of AI assistance

Claude (Anthropic) was used to help structure the `read_ausplay_section` helper function, discuss the rationale for opportunity-score weights, and suggest converting K-Means cluster labels to strings before plotting so Plotly renders discrete colours. All analytical decisions, parameter choices, and interpretations are my own.


## Setup


In [1]:
import requests
import json
import re
import time
from pathlib import Path

import pandas as pd
import numpy as np

import plotly.express as px

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import minmax_scale

import warnings
warnings.filterwarnings('ignore')


**Library roles:**

- `requests` / `json` — Guardian API access and response parsing
- `re` — HTML tag stripping and text cleaning
- `pandas` / `numpy` — data wrangling and numerical operations
- `plotly.express` — interactive visualisation
- `TfidfVectorizer` — converts article text into weighted term-frequency vectors
- `KMeans`, `minmax_scale` — strategic clustering and feature normalisation


## 1. Data Collection — Guardian Australia API

The Guardian Open Platform provides free API access to article content. An API key is required; store yours in `data/guardian_key.txt` (excluded from version control).


In [2]:
# Store your Guardian API key in data/guardian_key.txt (add that file to .gitignore)
key_path = Path('../private/guardian_key.txt')
key = key_path.read_text(encoding='utf-8').strip() if key_path.exists() else None
if key is None:
    print('Warning: data/guardian_key.txt not found. API calls will fail.')
else:
    print(f'API key loaded ({len(key)} chars)')


API key loaded (36 chars)


### Query Design

Eight targeted queries are used, each restricted to the Guardian Australia sport section (`production-office=aus&section=sport`). Restricting to `section=sport` avoids general Brisbane city news and AFL season results that contaminated broader searches in the exploratory phase.

Articles are collected from **1 January 2025 onwards** to capture the growing lead-up conversation as Brisbane 2032 moves from planning into active preparation.

| Query | Strategic rationale |
|-------|---------------------|
| `brisbane 2032 olympic sports` | Core Olympic planning coverage |
| `youth participation olympic sport` | Community engagement and junior development |
| `school sport australia` | Structured school-level participation |
| `olympic pathways australia` | Athlete development pipelines |
| `community sport participation` | Grassroots sporting culture |
| `women sport olympics` | Gender-inclusive coverage (added after initial run showed male skew) |
| `junior sport australia` | Junior club and competition coverage |
| `media coverage olympic athletes` | Athlete visibility and narrative-building |


In [3]:
search_queries = [
    'brisbane 2032 olympic sports',
    'youth participation olympic sport',
    'school sport australia',
    'olympic pathways australia',
    'community sport participation',
    'women sport olympics',
    'junior sport australia',
    'media coverage olympic athletes'
]
search_queries


['brisbane 2032 olympic sports',
 'youth participation olympic sport',
 'school sport australia',
 'olympic pathways australia',
 'community sport participation',
 'women sport olympics',
 'junior sport australia',
 'media coverage olympic athletes']

In [4]:
def get_guardian_articles(search_string,
                          from_date='2025-01-01',
                          production_office='aus',
                          section='sport',
                          page_size=50):
    """Fetch up to page_size Guardian articles matching search_string.

    Returns a list of dicts with keys: query, title, date, article_text.
    HTML tags are stripped from article body text during collection.
    """
    if key is None:
        raise ValueError('Guardian API key not found. See data/guardian_key.txt')

    url = (
        'https://content.guardianapis.com/search'
        f'?q={search_string}'
        f'&section={section}'
        f'&production-office={production_office}'
        f'&from-date={from_date}'
        f'&show-fields=body'
        f'&page-size={page_size}'
        f'&api-key={key}'
    )

    response = requests.get(url)
    response.raise_for_status()
    results = response.json().get('response', {}).get('results', [])

    articles = []
    for r in results:
        body_html = r['fields'].get('body', '')
        articles.append({
            'query':        search_string,
            'title':        r.get('webTitle', ''),
            'date':         r.get('webPublicationDate', ''),
            'article_text': re.sub(r'<.*?>', '', body_html)  # strip HTML tags
        })

    print(f'{search_string}: {len(articles)} articles fetched')
    return articles


In [5]:
saved_path = Path('../src/guardian_sports_articles.json')

if saved_path.exists():
    # Reuse previously saved dataset — avoids repeated API calls
    articles_df = pd.read_json(saved_path)
    print(f'Loaded saved Guardian dataset: {len(articles_df)} articles')
else:
    all_articles = []
    for query in search_queries:
        all_articles.extend(get_guardian_articles(query))
        time.sleep(1)  # respect API rate limits
    articles_df = pd.DataFrame(all_articles)
    print(f'Fetched Guardian dataset: {len(articles_df)} articles')


Loaded saved Guardian dataset: 170 articles


### Deduplication

Multiple overlapping queries can return the same article. Duplicates are removed on `title` before vectorisation to prevent the TF-IDF matrix from double-counting any article's terms. In this run, the eight queries were sufficiently distinct that **zero duplicates were found** — each query reached a different slice of Guardian sport coverage.


In [6]:
print('Articles before deduplication:', len(articles_df))
articles_df = articles_df.drop_duplicates(subset='title').reset_index(drop=True)
print('Articles after deduplication: ', len(articles_df))

# Save so the API does not need to be called again
saved_path.parent.mkdir(parents=True, exist_ok=True)
articles_df.to_json(saved_path, orient='records', indent=4, date_format='iso')
print(f'Dataset saved to {saved_path}')


Articles before deduplication: 170
Articles after deduplication:  170
Dataset saved to ../src/guardian_sports_articles.json


---
## 2. Text Analytics — TF-IDF Vectorisation

### Text Preprocessing

Article text is cleaned before vectorisation to reduce vocabulary noise:

- **Lowercase** — ensures `Olympic` and `olympic` are treated as the same token
- **Remove punctuation and numbers** — reduces vocabulary to alphabetic terms only
- **Strip extra whitespace** — prevents empty tokens entering the vectoriser

HTML tags were already stripped during collection (`re.sub(r'<.*?>', '', body)`).


In [7]:
def clean_article_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

articles_df['cleaned_text'] = articles_df['article_text'].apply(clean_article_text)
articles_df[['title', 'cleaned_text']].head(3)


,title,cleaned_text
0,Andrew Dillon reveals AFL’s Olympic-sized ambi...,on the eve of the season the afl boss andrew d...
1,‘A big call for the IOC’: is the fight over Ol...,an act of god cancelled rowing at the first ol...
2,"Believe, belong, become, boring, bizarre: Bris...",if you typed the words believe belong and beco...


### TF-IDF Vectorisation

TF-IDF (Term Frequency–Inverse Document Frequency) converts article text into a numeric matrix where each value reflects how distinctive a term is to a particular article relative to the whole corpus. Words appearing in nearly every article (`sport`, `australian`) receive low scores; words distinctive to a sub-set of articles receive high scores.

This matrix serves two purposes:
1. **Media term landscape** — mean TF-IDF across all articles shows which topics dominate coverage
2. **Input to K-Means clustering** — article vectors group articles by topic similarity

**Parameter choices:**

| Parameter | Value | Reason |
|-----------|-------|--------|
| `max_df` | 0.80 | Remove terms in >80% of articles (corpus-wide filler missed by stop-list) |
| `min_df` | 2 | Remove terms in only one article (likely typos or unique proper nouns) |
| `max_features` | 1000 | Cap vocabulary to keep matrix manageable and clustering interpretable |
| `stop_words` | custom | Standard English list + journalism noise words (`said`, `gmt`, etc.) |


In [8]:
# Initial vectorisation with standard English stop-words
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.80, min_df=2, max_features=1000, stop_words='english'
)
tfidf_matrix = tfidf_vectorizer.fit_transform(articles_df['cleaned_text'])
feature_names = tfidf_vectorizer.get_feature_names_out()

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}  (rows = articles, columns = features)')

# Inspect top terms to identify any remaining noise words
tfidf_scores = np.asarray(tfidf_matrix.mean(axis=0)).ravel()
pd.DataFrame({'term': feature_names, 'mean_tfidf': tfidf_scores}) \
  .sort_values('mean_tfidf', ascending=False).head(15)


TF-IDF matrix shape: (170, 1000)  (rows = articles, columns = features)


,term,mean_tfidf
764,said,0.062897
990,world,0.051699
826,sport,0.051284
9,afl,0.048526
42,australian,0.044082
348,gout,0.042873
445,just,0.041562
770,says,0.040044
288,final,0.036326
95,brisbane,0.034391


### Refining the Vocabulary

The initial top terms include generic journalism phrases (`said`, `says`, `gmt`) that appear frequently but carry no sport-topic information. A custom stop-list removes them before producing the final vocabulary used in all downstream analysis.


In [9]:
custom_stopwords = [
    'said', 'says', 'gmt', 'just', 'year', 'years',
    'new', 'time', 'people', 'like', 'including',
    'week', 'day', 'days'
]

english_stopwords = list(TfidfVectorizer(stop_words='english').get_stop_words())
all_stopwords    = english_stopwords + custom_stopwords
print(f'Total stop-words: {len(all_stopwords)} '
      f'({len(english_stopwords)} standard + {len(custom_stopwords)} custom)')

refined_tfidf  = TfidfVectorizer(
    max_df=0.80, min_df=2, max_features=1000, stop_words=all_stopwords
)
refined_matrix   = refined_tfidf.fit_transform(articles_df['cleaned_text'])
refined_features = refined_tfidf.get_feature_names_out()

refined_scores  = np.asarray(refined_matrix.mean(axis=0)).ravel()
refined_summary = (
    pd.DataFrame({'term': refined_features, 'mean_tfidf': refined_scores})
    .sort_values('mean_tfidf', ascending=False)
)
refined_summary.head(20)


Total stop-words: 332 (318 standard + 14 custom)


,term,mean_tfidf
991,world,0.052533
827,sport,0.052532
9,afl,0.049685
43,australian,0.045085
350,gout,0.043516
292,final,0.037010
97,brisbane,0.035282
326,games,0.035026
36,athletes,0.034599
607,olympic,0.033845


In [10]:
fig = px.bar(
    refined_summary.head(20),
    x='mean_tfidf', y='term', orientation='h',
    title='Top 20 TF-IDF Terms Across Guardian Australia Sport Articles',
    labels={'mean_tfidf': 'Mean TF-IDF score', 'term': 'Term'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()


### Media Landscape Interpretation

The refined term list confirms the corpus is centred on the right topic area — `olympic`, `games`, `athletes`, and `brisbane` all appear prominently. `afl` and `final` reflect articles where the Guardian's sport section co-locates Olympic planning content with AFL season coverage, a reminder that even a sport-restricted search picks up non-Olympic material.

The most important observation: **specific mid-tier sport names (hockey, volleyball, taekwondo) do not appear in the top 20 terms at all**, despite the searches being deliberately broad. This corroborates the media-visibility gap that motivates the opportunity score below.


---
## 3. Structured Analysis — AusPlay Participation Data

To answer the research question we need to know where children are *actually* participating, not just what the media is discussing. Four AusPlay sheets provide complementary evidence:

| Sheet | Content | Why included |
|-------|---------|-------------|
| 1 | Adult participation rates by age and gender | Captures 15-17 and 18-24 bands for pathway analysis |
| 2 | Child participation rates by age group | Core participation evidence |
| 5 | Organised child participation (club and school) | Shows whether participation has formal structure |
| 7 | Year-on-year change in child participation | Captures momentum, not just current size |

Participation *rates* (proportions) are used rather than estimated counts so sports can be compared on a common scale. A child rate of 0.236 for Swimming means approximately **24 in every 100 Australian children** participate — immediately comparable to Hockey's rate of 0.010 (about 1 in every 100).

> **Ethical note:** AusPlay data is survey-based. Very small rate values (below ~0.003) carry significant uncertainty and should not drive investment decisions in isolation.


In [11]:
# ausplay_file = '../src/ausplay_by_sport_2024_25.xlsx'
ausplay_file = "../src/C4S-AusPlay-By-Sport-Data-Tables-13-November-2025.xlsx"


In [12]:
def read_ausplay_section(sheet_name, columns, section='rate'):
    """Load one section (rate or count) from an AusPlay sheet.

    Each AusPlay sheet contains two stacked tables separated by a row
    containing 'Estimated number of participants'. The top table holds
    participation rates; the bottom holds estimated counts. Column indices
    are zero-based positions in the raw Excel sheet.
    """
    raw = pd.read_excel(ausplay_file, sheet_name=sheet_name, header=None)

    marker_rows = raw.index[
        raw[1].astype(str).str.contains('Estimated number of participants', na=False)
    ].tolist()

    if section == 'rate':
        end_row = marker_rows[0] - 1 if marker_rows else len(raw) - 1
        data    = raw.loc[15:end_row].copy()
    else:
        start_row = marker_rows[0] + 1 if marker_rows else 15
        data      = raw.loc[start_row:].copy()

    data = data[data[0].notna()]
    data = data[~data[0].astype(str).str.startswith(('NB.', '*', '#'))]

    output = pd.DataFrame({'sport': data[0].astype(str).str.strip()})
    for col_name, col_idx in columns.items():
        output[col_name] = pd.to_numeric(data[col_idx], errors='coerce')

    first_metric = list(columns.keys())[0]
    return (
        output
        .dropna(subset=[first_metric])
        .drop_duplicates(subset='sport')
        .reset_index(drop=True)
    )


In [13]:
adult_rates = read_ausplay_section(
    '1',
    {'adult_total': 1, 'adult_male': 3, 'adult_female': 4,
     'adult_15_17': 6, 'adult_18_24': 7},
    section='rate'
)

child_rates = read_ausplay_section(
    '2',
    {'child_total': 1, 'child_male': 3, 'child_female': 4,
     'child_0_4': 6, 'child_5_8': 7, 'child_9_11': 8, 'child_12_14': 9},
    section='rate'
)

child_organised = read_ausplay_section(
    '5',
    {'child_organised_total': 1,
     'child_sports_club': 2,
     'child_educational_institution': 7},
    section='rate'
)

child_yoy = read_ausplay_section(
    '7',
    {'child_2023_24': 1, 'child_2024_25': 3},
    section='rate'
)

print(f'adult_rates:     {len(adult_rates)} sports')
print(f'child_rates:     {len(child_rates)} sports')
print(f'child_organised: {len(child_organised)} sports')
print(f'child_yoy:       {len(child_yoy)} sports')
child_rates.head()


adult_rates:     142 sports
child_rates:     142 sports
child_organised: 142 sports
child_yoy:       142 sports


,sport,child_total,child_male,child_female,child_0_4,child_5_8,child_9_11,child_12_14
0,Adventure racing,0.002239,0.002137,0.002360,0.002169,0.003882,0.001370,0.001107
1,Air sports,0.002589,0.003689,0.001443,0.002729,0.003752,0.000000,0.003466
2,Archery,0.001481,0.001927,0.001018,0.001160,0.000870,0.001194,0.003028
3,"Athletics, track and field",0.032402,0.031589,0.033213,0.012942,0.041321,0.045589,0.037192
4,Australian football,0.063026,0.091643,0.032581,0.013615,0.094516,0.083491,0.076826


### Olympic Sport Selection and Data Merge

AusPlay covers many activities that are not on the Olympic programme. Only sports with a confirmed or long-standing Olympic presence are retained. AusPlay activity names are used verbatim to preserve the join key.

The four sheets are merged on sport name. Year-on-year growth percentage is derived from the raw rate change, normalised by the prior-year rate so different-sized sports are comparable.

The `adult_15_17` and `child_12_14` columns are deliberately included: children currently aged 12-14 would be 18-21 in 2032 — old enough to be competing. This is the most direct peer connection the ASC can draw on for a 2032 narrative.


In [14]:
olympic_sports = [
    'Archery', 'Athletics, track and field', 'Badminton', 'Basketball',
    'BMX', 'Boxing', 'Canoeing', 'Cycling', 'Diving', 'Equestrian',
    'Fencing', 'Football/soccer', 'Golf', 'Gymnastics', 'Hockey',
    'Judo', 'Rowing', 'Rugby union', 'Sailing', 'Shooting',
    'Skate sports', 'Softball', 'Surfing', 'Swimming', 'Table tennis',
    'Taekwondo/Taekwon-do', 'Tennis', 'Triathlon',
    'Volleyball (indoor and outdoor)', 'Water polo',
    'Weight lifting', 'Wrestling'
]

participation_df = (
    child_rates[child_rates['sport'].isin(olympic_sports)]
    .merge(adult_rates,     on='sport', how='left')
    .merge(child_yoy,       on='sport', how='left')
    .merge(child_organised, on='sport', how='left')
)

participation_df['child_growth']     = (
    participation_df['child_2024_25'] - participation_df['child_2023_24']
)
participation_df['child_growth_pct'] = np.where(
    participation_df['child_2023_24'] > 0,
    participation_df['child_growth'] / participation_df['child_2023_24'],
    np.nan
)

print(f'Olympic sports matched in AusPlay: {len(participation_df)}')
participation_df[
    ['sport', 'child_total', 'child_12_14', 'child_growth_pct', 'adult_15_17']
].sort_values('child_total', ascending=False).head(15)


Olympic sports matched in AusPlay: 29


,sport,child_total,child_12_14,child_growth_pct,adult_15_17
20,Swimming,0.235637,0.107515,0.027362,0.186415
10,Football/soccer,0.145308,0.159820,0.108035,0.181312
3,Basketball,0.070958,0.123129,0.051777,0.160229
12,Gymnastics,0.070880,0.027492,0.056409,0.022217
23,Tennis,0.039848,0.059045,0.016338,0.089282
1,"Athletics, track and field",0.032402,0.037192,0.090614,0.049090
6,Cycling,0.017208,0.015196,-0.391789,0.059223
13,Hockey,0.009600,0.018905,0.263071,0.011286
22,Taekwondo/Taekwon-do,0.008274,0.008132,0.011600,0.007302
2,Badminton,0.007505,0.013893,-0.229899,0.073870


In [15]:
fig = px.bar(
    participation_df.sort_values('child_total', ascending=False).head(15),
    x='child_total', y='sport', orientation='h',
    title='Top 15 Olympic-Aligned Sports by Child Participation Rate (AusPlay 2024-25)',
    labels={'child_total': 'Child participation rate', 'sport': 'Sport'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()


---
## 4. Linking Media Visibility to Participation

Media visibility is measured by counting the number of Guardian articles in the corpus that mention each sport by name. A sport-keyword dictionary maps each Olympic sport to one or more search terms to handle common synonyms (e.g. `canoe`, `canoeing`, `kayak`). Word-boundary matching (`\\b`) prevents `row` from accidentally matching inside `rowing`.

This metric intentionally measures *relative* visibility within one corpus rather than absolute Australian media coverage.

> **Ethical note:** Guardian coverage reflects one outlet's editorial priorities. Sports popular in communities under-represented in mainstream English-language media (martial arts, canoeing, water polo) may be systematically undercounted. These findings should be validated against broader media datasets before informing funding decisions.


In [16]:
sport_search_terms = {
    'Archery':                          ['archery'],
    'Athletics, track and field':        ['athletics', 'track and field'],
    'Badminton':                         ['badminton'],
    'Basketball':                        ['basketball'],
    'BMX':                               ['bmx'],
    'Boxing':                            ['boxing'],
    'Canoeing':                          ['canoe', 'canoeing', 'kayak'],
    'Cycling':                           ['cycling', 'cyclist'],
    'Diving':                            ['diving'],
    'Equestrian':                        ['equestrian', 'horse riding'],
    'Fencing':                           ['fencing'],
    'Football/soccer':                   ['football', 'soccer'],
    'Golf':                              ['golf'],
    'Gymnastics':                        ['gymnastics'],
    'Hockey':                            ['hockey'],
    'Judo':                              ['judo'],
    'Rowing':                            ['rowing'],
    'Rugby union':                       ['rugby union', 'rugby sevens'],
    'Sailing':                           ['sailing'],
    'Shooting':                          ['shooting'],
    'Skate sports':                      ['skateboarding', 'skate'],
    'Softball':                          ['softball'],
    'Surfing':                           ['surfing'],
    'Swimming':                          ['swimming', 'swimmer'],
    'Table tennis':                      ['table tennis'],
    'Taekwondo/Taekwon-do':              ['taekwondo'],
    'Tennis':                            ['tennis'],
    'Triathlon':                         ['triathlon'],
    'Volleyball (indoor and outdoor)':   ['volleyball'],
    'Water polo':                        ['water polo'],
    'Weight lifting':                    ['weightlifting', 'weight lifting'],
    'Wrestling':                         ['wrestling'],
}

# Combine title and full article text into one searchable string per article
article_corpus = (
    articles_df['title'].fillna('') + ' ' + articles_df['article_text'].fillna('')
).str.lower()

media_rows = []
for sport, terms in sport_search_terms.items():
    pattern = r'\b(?:' + '|'.join(re.escape(t) for t in terms) + r')\b'
    media_rows.append({
        'sport': sport,
        'media_mentions': int(article_corpus.str.contains(pattern, regex=True).sum())
    })

media_df = pd.DataFrame(media_rows)
media_df.sort_values('media_mentions', ascending=False).head(15)


,sport,media_mentions
11,Football/soccer,54
1,"Athletics, track and field",34
23,Swimming,25
26,Tennis,20
8,Diving,10
17,Rugby union,9
12,Golf,8
14,Hockey,6
16,Rowing,5
3,Basketball,5


### Opportunity Score

All four participation-and-visibility dimensions are normalised to [0, 1] using min-max scaling so they contribute on an equal footing before weighting. The composite opportunity score applies the following weights:

| Dimension | Weight | Rationale |
|-----------|--------|-----------|
| Overall child participation | **40%** | Largest driver — an existing youth base is the prerequisite for fan conversion |
| 12-14 participation | **25%** | Proxy for near-future Olympians; this age group is also entering independent social media use |
| Year-on-year growth | **20%** | Sports with growing child bases will compound investment returns |
| Low media visibility (inverted) | **15%** | Highlights where ASC investment is additive; lower weight because the Guardian corpus is a limited sample |


In [17]:
combined_df = participation_df.merge(media_df, on='sport', how='left')
combined_df['media_mentions'] = combined_df['media_mentions'].fillna(0)

combined_df['participation_score'] = minmax_scale(combined_df['child_total'])
combined_df['older_child_score']   = minmax_scale(combined_df['child_12_14'])
combined_df['growth_score']        = minmax_scale(combined_df['child_growth_pct'].fillna(0))
combined_df['visibility_score']    = minmax_scale(combined_df['media_mentions'])

combined_df['opportunity_score'] = (
    0.40 * combined_df['participation_score']
    + 0.25 * combined_df['older_child_score']
    + 0.20 * combined_df['growth_score']
    + 0.15 * (1 - combined_df['visibility_score'])  # inverted: low visibility = high opportunity
)

opportunity_df = combined_df.sort_values('opportunity_score', ascending=False)[[
    'sport', 'child_total', 'child_12_14', 'child_growth_pct',
    'media_mentions', 'opportunity_score'
]]
opportunity_df.head(12)


,sport,child_total,child_12_14,child_growth_pct,media_mentions,opportunity_score
20,Swimming,0.235637,0.107515,0.027362,25,0.736430
10,Football/soccer,0.145308,0.159820,0.108035,54,0.592811
3,Basketball,0.070958,0.123129,0.051777,5,0.539353
12,Gymnastics,0.070880,0.027492,0.056409,1,0.400895
14,Judo,0.001555,0.002046,1.113655,0,0.355206
23,Tennis,0.039848,0.059045,0.016338,20,0.340747
25,Volleyball (indoor and outdoor),0.007388,0.021231,0.183810,2,0.293643
13,Hockey,0.009600,0.018905,0.263071,6,0.290823
9,Fencing,0.000948,0.001926,0.439345,0,0.284380
4,BMX,0.001626,0.002786,0.217724,0,0.264002


**Interpreting the ranking:** The score is a decision-support tool, not a definitive priority list. Sports with `child_total` below 0.003 (Judo, Fencing, BMX) appear high due to extreme percentage growth from very small bases — mathematically a rate doubling from 0.0008 to 0.0016 produces 100% growth but represents very few additional children. The strategically meaningful targets — those with both a material participation base and a high score — are: **Gymnastics** (score 0.400, only 1 media mention despite child_total 0.071), **Volleyball** (score 0.293, 18% growth), **Hockey** (score 0.288, 26% growth), and **Taekwondo** (score 0.262).


In [18]:
fig = px.scatter(
    combined_df,
    x='media_mentions', y='child_total',
    size='child_12_14', color='child_growth_pct',
    hover_name='sport',
    title='Youth Participation vs Media Visibility (size = 12-14 rate, colour = YoY growth)',
    labels={
        'media_mentions':    'Guardian article mentions',
        'child_total':       'Child participation rate',
        'child_growth_pct':  'Year-on-year growth',
        'child_12_14':       '12-14 participation rate'
    },
    color_continuous_scale='Tealrose'
)
fig.show()


Sports with high youth participation do not always receive proportional media attention. This plot makes the gap visible: the upper-left quadrant (high participation, low mentions) is where strategic investment is most additive. Gymnastics, Hockey, and Volleyball all occupy this space — their large bubble sizes confirm the 12-14 age band is well-represented, meaning these sports also satisfy the near-future Olympian criterion.


In [19]:
fig = px.bar(
    opportunity_df.head(15),
    x='opportunity_score', y='sport', orientation='h',
    title='Top 15 Olympic Sports by ASC Opportunity Score',
    labels={'opportunity_score': 'Opportunity score (0–1)', 'sport': 'Sport'},
    color='opportunity_score', color_continuous_scale='Teal'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False)
fig.show()


---
## 5. K-Means Strategic Segmentation

K-Means clustering groups Olympic sports by their combined profile across child participation, 12-14 participation, year-on-year growth, and media visibility. The goal is exploratory segmentation to produce directly actionable strategic groups rather than prediction.

Four clusters were selected to provide an interpretable strategic segmentation. The objective was exploratory grouping rather than optimising predictive performance. Cluster counts were evaluated for interpretability and separation, with four producing the clearest actionable segments. All features are normalised before clustering so no single variable dominates by scale.


In [20]:
cluster_features = combined_df[
    ['child_total', 'child_12_14', 'child_growth_pct', 'media_mentions']
].copy()
cluster_features['child_growth_pct'] = cluster_features['child_growth_pct'].fillna(0)

cluster_features_scaled = pd.DataFrame(
    minmax_scale(cluster_features),
    columns=cluster_features.columns,
    index=combined_df.index
)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
combined_df['cluster'] = kmeans.fit_predict(cluster_features_scaled).astype(str)

cluster_summary = (
    combined_df
    .groupby('cluster')
    .agg(
        n_sports               = ('sport',            'count'),
        avg_child_participation = ('child_total',      'mean'),
        avg_12_14_participation = ('child_12_14',      'mean'),
        avg_growth_pct          = ('child_growth_pct', 'mean'),
        avg_media_mentions      = ('media_mentions',   'mean'),
        sports                  = ('sport', lambda v: ', '.join(v))
    )
    .reset_index()
)
cluster_summary


,cluster,n_sports,avg_child_participation,avg_12_14_participation,avg_growth_pct,avg_media_mentions,sports
0,0,8,0.005102,0.006861,-0.410054,3.625000,"Archery, Badminton, Cycling, Equestrian, Golf,..."
1,1,2,0.190473,0.133667,0.067699,39.500000,"Football/soccer, Swimming"
2,2,16,0.007508,0.007726,0.198867,2.375000,"BMX, Boxing, Diving, Fencing, Gymnastics, Hock..."
3,3,3,0.047736,0.073122,0.052910,19.666667,"Athletics, track and field, Basketball, Tennis"


In [21]:
fig = px.scatter(
    combined_df,
    x='media_mentions', y='child_total',
    color='cluster', size='opportunity_score',
    hover_name='sport',
    title='K-Means Clusters: Olympic Sports by Participation and Media Visibility',
    labels={
        'media_mentions': 'Guardian article mentions',
        'child_total':    'Child participation rate',
        'cluster':        'Cluster'
    }
)
fig.show()


### Cluster Interpretation and Strategic Response

---

**Cluster 1 — High participation, high visibility** *(Football/soccer, Swimming)*  
These two sports dominate both AusPlay rates and Guardian coverage (avg 39.5 mentions). They already have strong fan pipelines, well-funded athlete programmes, and extensive media presence. The ASC role here is **maintenance, not new investment** — both are saturated opportunity rather than untapped potential.

---

**Cluster 3 — Moderate participation, moderate visibility** *(Athletics, Basketball, Tennis)*  
Meaningful youth participation (avg child rate ~0.048) and genuine media attention (avg 22 mentions). Not invisible, but not saturated. The ASC could **leverage existing media presence to deepen youth engagement** — particularly Athletics, which shows 9% year-on-year growth and a 12-14 rate of 0.037. A targeted athlete ambassador programme could connect junior club members with a realistic pathway to 2032.

---

**Cluster 2 — Primary strategic target** *(16 sports including Hockey, Volleyball, Gymnastics, Surfing, Taekwondo)*  
The largest cluster and the most important finding. These 16 sports share modest but real child participation (avg rate 0.0075) and very low media presence (avg 2.4 Guardian mentions). The positive average growth rate (0.199) signals community interest is building organically — the audience is growing without any media amplification. Investment here is genuinely additive:

- Children currently participating at ages 12-14 (Hockey, Gymnastics) are on a realistic timeline to compete at Brisbane 2032
- The fan base already exists as junior participants — it simply is not connected to the Olympic programme
- These children are entering social media age now; ASC digital content and school outreach would reach an audience forming its media habits before 2032

---

**Cluster 0 — Declining small sports** *(Archery, Badminton, Cycling, Golf and others)*  
Negative average growth (-0.41) — child participation is shrinking. Cycling has a meaningful absolute base but is losing ground. The ASC should **monitor rather than prioritise** this group. Understanding why participation is falling — cost, access, cultural shifts — is the necessary first step before any investment decision.


---
## Ethical Reflection

This analysis uses aggregated public datasets and does not identify individual participants. Several ethical considerations are directly relevant to how the findings should be used:

**Data uncertainty**  
AusPlay data is survey-based with a margin of error, particularly for smaller sports. Very small participation rates (below ~0.003) carry significant uncertainty and should not drive investment decisions in isolation.

**Media sample bias**  
Guardian coverage reflects one outlet's editorial priorities. Sports popular in communities under-represented in mainstream English-language media may be systematically undercounted in the visibility metric. Low media visibility may reflect structural media bias rather than a genuine absence of community interest.

**Conflating visibility with value**  
Media visibility is not the same as cultural importance, athlete potential, or social benefit. A sport ranked lower by the opportunity score is not less valuable; it may simply be less legible to one outlet's editorial algorithm.

**Equity in sport access**  
Child participation rates are shaped by cost, geography, facilities, disability access, and school PE curricula. High participation may reflect cost-accessibility rather than intrinsic interest. Low participation may indicate barriers the ASC could address — not a reason to deprioritise a sport.

**Gender reporting gap**  
AusPlay notes that non-binary and self-described gender responses are included in totals but not reported separately due to sample size constraints. Gender-based comparisons in this analysis do not represent all children participating.

> These findings should be used to guide stakeholder consultation and qualitative research rather than to make final resourcing decisions in isolation.


---
## Findings and Strategic Recommendations

This analysis joined two evidence streams — structured AusPlay participation data and semi-structured Guardian media text — to identify where the ASC can most effectively build fan support ahead of Brisbane 2032.

---

### Finding 1: Near-future Olympians in the 12-14 band

Children currently aged 12-14 would be 18-21 at Brisbane 2032 — old enough to compete. Football/soccer had the highest 12-14 rate (0.160) but already receives 54 Guardian mentions and needs no additional ASC attention in this regard. The more valuable peer connection opportunity lies in sports with strong 12-14 rates and far less commercial attention: **Athletics** (0.037, growing 9%), **Hockey** (0.019, growing 26%), and **Volleyball** (0.021, growing 18%).

---

### Finding 2: The participation-visibility gap

Applying a minimum participation threshold to filter out small-sport statistical noise (`child_total ≥ 0.003`), the primary investment targets are:

| Sport | Opportunity score | Media mentions | Child participation | YoY growth |
|-------|:-----------------:|:--------------:|:-------------------:|:----------:|
| Gymnastics | 0.400 | 1 | 0.071 | +6% |
| Volleyball | 0.294 | 2 | 0.007 | +18% |
| Hockey | 0.291 | 6 | 0.010 | +26% |
| Taekwondo | 0.262 | 0 | 0.008 | +1% |

Each of these sports has tens of thousands of young participants who are receiving almost no Olympic narrative in Australian media. Connecting junior participants to national athletes through school programmes and digital content would convert existing participation into fan engagement at relatively low marginal cost.

---

### Finding 3: Early adolescents and media habits

The 12-14 age band is entering independent social media use now. Hockey's 12-14 rate (0.019) received only 6 Guardian mentions across 170 articles — by comparison, Diving received 10 mentions with a `child_total` of just 0.003. Media attention is not proportional to youth participation. The K-Means Cluster 2 sports are the highest-leverage targets for a social media strategy because their audience is forming media habits before 2032.

---

### Strategic summary

The strongest value of this analysis is that it joins two streams that are individually insufficient: high participation without media presence means no narrative to convert participants into fans; high media presence without participation means a passive audience rather than an engaged one. The opportunity score and K-Means clusters identify precisely where those two conditions are misaligned — and that misalignment is where ASC intervention is most likely to produce measurable change by 2032.

These findings should be treated as a starting point for stakeholder consultation, not a final investment plan. Qualitative research with junior participants and coaches in the identified sports would validate whether the media visibility gap reflects a genuine absence of content or a limitation of the single-outlet corpus used here.
